# Final Plots for TAA Paper

In [1]:
import pandas as pd
import altair as alt
import numpy as np


from dataloader import load_df, load_perframe_pervalue, PARAM_LABELS, PARAMS, RESOLUTIONS, RESOLUTION_LABELS

DATA_PATH = "dataset.json"  
PARAMETERIZATION_PATH = "video_paramaterization/video_stats.csv"
COMPUTE_PATH = "compute_time/results-TAA.csv"


In [2]:
df_centered = load_df(DATA_PATH, center=True)
df_raw = load_df(DATA_PATH, center=False)
df_pf = load_perframe_pervalue(DATA_PATH)

/Users/mariasilva/Documents/PerceptualTAA/dataloader.py:103: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(_center_group)


## Parameter Analysis

#### Range Table +- std

In [3]:
UNREAL_DEFAULTS = {
        'alpha_weight': 0.04,
        'hist_percent': 100,
        'filter_size': 1,
        'num_samples': 8
    }

In [4]:
def compute_range_table(df, score_col):
    range_per_scene = (
        df.groupby(['scene', 'resolution', 'parameter'])[score_col]
        .agg(lambda x: x.max() - x.min())
        .reset_index()
        .rename(columns={score_col: 'score_range'})
    )
    return (
        range_per_scene
        .groupby(['resolution', 'parameter'])['score_range']
        .agg(lambda x: f"{x.mean():.2f} ± {x.std():.2f}")
        .unstack('parameter')
        .rename(columns=PARAM_LABELS)
        .sort_index(ascending=False)
        .rename(index=RESOLUTION_LABELS)
    )

table_centered = compute_range_table(df_centered, 'score_centered')
table_raw      = compute_range_table(df_raw, 'score')

print("RAW")
display(table_raw)

RAW


parameter,Alpha weight,Filter size,History %,Num samples
resolution,,,,
100%,2.71 ± 2.82,0.15 ± 0.25,1.37 ± 1.19,0.29 ± 0.26
75%,2.79 ± 2.46,0.15 ± 0.25,1.43 ± 1.27,0.18 ± 0.25
50%,3.66 ± 3.27,0.14 ± 0.23,1.36 ± 1.17,0.16 ± 0.23
25%,6.39 ± 6.89,0.13 ± 0.18,1.46 ± 1.04,0.16 ± 0.20


In [5]:
def compute_recommendations(df, resolution_labels=RESOLUTION_LABELS):
    recommendations = {}
    results = {}
    
    for param, default_val in UNREAL_DEFAULTS.items():
        param_df = df[df['parameter'] == param].copy()
        mean_scores = (
            param_df.groupby(['resolution', 'value'])['score']
            .mean()
            .reset_index()
        )
        # Optimal (argmax) per resolution
        optimal = mean_scores.loc[mean_scores.groupby('resolution')['score'].idxmax()].copy()
        optimal = optimal.rename(columns={'value': 'optimal_value', 'score': 'optimal_score'})
        # Baseline score at Unreal default
        baseline = (
            mean_scores[mean_scores['value'] == default_val]
            .rename(columns={'score': 'baseline_score'})
            [['resolution', 'baseline_score']]
        )
        # Join and compute gain
        result = optimal.merge(baseline, on='resolution')
        result['gain'] = result['optimal_score'] - result['baseline_score']
        result['resolution'] = result['resolution'].map(resolution_labels)
        result = result.sort_values('resolution', ascending=False)
        
        results[param] = result[['resolution', 'baseline_score', 'optimal_value', 'optimal_score', 'gain']].round(2)
        
        # {resolution_label: optimal_value} for the plot
        recommendations[param] = dict(zip(result['resolution'], result['optimal_value']))
    
    return recommendations, results

RECOMMENDATIONS, rec_results = compute_recommendations(df_raw)

In [6]:
print(RECOMMENDATIONS)

{'alpha_weight': {'75%': 0.5, '50%': 0.2, '25%': 0.2, '100%': 0.6}, 'hist_percent': {'75%': 200.0, '50%': 200.0, '25%': 150.0, '100%': 200.0}, 'filter_size': {'75%': 0.95, '50%': 0.95, '25%': 1.0, '100%': 0.9}, 'num_samples': {'75%': 64.0, '50%': 64.0, '25%': 8.0, '100%': 8.0}}


#### Gain Table (in terms of global rec, or indiv rec)

In [7]:
def plot_gain_table(df, parameter, mode='recs'):
    """
    Plot gain table for a given parameter.
    
    Parameters
    ----------
    df : DataFrame
        Output of load_df(..., center=False)
    parameter : str
        One of 'alpha_weight', 'hist_percent', 'filter_size', 'num_samples'
    mode : str
        'recs'     : gain = score(our recommendation) - score(unreal default)
        'oracle'   : gain = score(best per scene)     - score(unreal default)
    """
    param_df = df[df['parameter'] == parameter].copy()
    baseline_val = UNREAL_DEFAULTS[parameter]

    # Baseline score per (scene, resolution)
    baselines = (
        param_df[param_df['value'] == baseline_val]
        .set_index(['scene', 'resolution'])['score']
        .rename('baseline_score')
    )

    if mode == 'oracle':
        target = (
            param_df.groupby(['scene', 'resolution'])['score']
            .max()
            .rename('target_score')
        )
    elif mode == 'recs':
        # Look up recommended value for this param at each resolution
        def get_rec_score(group):
            scene, res = group.name
            rec_val = RECOMMENDATIONS[parameter][RESOLUTION_LABELS[res]]
            row = group[group['value'] == rec_val]
            if row.empty:
                return pd.Series({'scene': scene, 'resolution': res, 'target_score': float('nan')})
            return pd.Series({'scene': scene, 'resolution': res, 'target_score': row['score'].iloc[0]})

        target = (
            param_df.groupby(['scene', 'resolution'])
            .apply(get_rec_score)
            .set_index(['scene', 'resolution'])['target_score']
        )
    else:
        raise ValueError("mode must be 'recs' or 'oracle'")

    # Combine and compute gain
    scene_gains = pd.merge(target, baselines, left_index=True, right_index=True).reset_index()
    scene_gains['gain'] = scene_gains['target_score'] - scene_gains['baseline_score']

    # Aggregate across scenes
    gain_summary = (
        scene_gains.groupby('resolution')['gain']
        .agg([
            ('Avg Gain', 'mean'),
            ('Best Case (Max)', 'max'),
            ('Worst Case (Min)', 'min'),
            ('Std Dev', 'std'),
        ])
        .sort_index(ascending=False)
        .rename(index=RESOLUTION_LABELS)
        .round(2)
    )

    mode_label = "Per-Scene Oracle" if mode == 'oracle' else "Recommended Value"
    print(f"Gain Table — {PARAM_LABELS[parameter]} [{mode_label} vs Unreal Default]")
    # display(gain_summary)
    return gain_summary

In [8]:
# Using our recommendations
plot_gain_table(df_raw, 'alpha_weight', mode='recs')

Gain Table — Alpha weight [Recommended Value vs Unreal Default]


/var/folders/wh/dwptv6ss1nzbsg6xw_2p4c9h0000gn/T/ipykernel_10192/4107186888.py:43: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(get_rec_score)


,Avg Gain,Best Case (Max),Worst Case (Min),Std Dev
resolution,,,,
100%,2.02,7.54,-0.26,2.32
75%,1.89,6.54,-0.05,1.95
50%,1.50,4.66,0.10,1.30
25%,1.25,3.96,0.12,0.98


In [9]:
# Using oracle (individual best per scene)
plot_gain_table(df_raw, 'alpha_weight', mode='oracle')

Gain Table — Alpha weight [Per-Scene Oracle vs Unreal Default]


,Avg Gain,Best Case (Max),Worst Case (Min),Std Dev
resolution,,,,
100%,2.11,7.54,0.14,2.29
75%,1.94,6.54,0.13,1.96
50%,1.67,4.66,0.12,1.46
25%,1.27,3.96,0.13,0.97


In [10]:
plot_gain_table(df_raw, 'hist_percent', mode='recs')

Gain Table — History % [Recommended Value vs Unreal Default]


/var/folders/wh/dwptv6ss1nzbsg6xw_2p4c9h0000gn/T/ipykernel_10192/4107186888.py:43: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(get_rec_score)


,Avg Gain,Best Case (Max),Worst Case (Min),Std Dev
resolution,,,,
100%,1.36,4.16,0.07,1.19
75%,1.41,4.31,0.03,1.29
50%,1.22,4.31,-1.11,1.30
25%,0.38,2.35,-2.50,0.98


In [11]:
plot_gain_table(df_raw, 'hist_percent', mode='oracle')

Gain Table — History % [Per-Scene Oracle vs Unreal Default]


,Avg Gain,Best Case (Max),Worst Case (Min),Std Dev
resolution,,,,
100%,1.37,4.16,0.07,1.19
75%,1.43,4.31,0.09,1.27
50%,1.31,4.31,0.08,1.21
25%,0.68,3.31,0.00,0.97


#### Response Curves
- note the gain here shows rec value gain! (on average, always choose our default... not the best for the scene)

In [12]:
res_order  = ['100%', '87%', '71%', '50%']
res_colors = ['#2a7db5', '#2ca05a', '#e07b3d', '#d4504a']
color_scale = alt.Scale(domain=res_order, range=res_colors)

In [ ]:
def plot_response_curves(input_df, score_col, y_title, plot_title, y_min=None, y_max=None, annotation_mode='recs'):
    display_map = {'100%': '100%', '87%': '75%', '71%': '50%', '50%': '25%'}
    reverse_map = {v: k for k, v in display_map.items()}

    agg = (
        input_df.groupby(['parameter', 'resolution', 'value'])[score_col]
        .agg(['mean', 'std', 'count'])
        .reset_index()
    )
    agg['resolution'] = agg['resolution'].astype(str).str.replace('%', '') + '%'
    agg['ci_low']  = agg['mean'] - agg['std']
    agg['ci_high'] = agg['mean'] + agg['std']

    _y_min = y_min if y_min is not None else agg['ci_low'].min() - 0.5
    _y_max = y_max if y_max is not None else agg['ci_high'].max() + 0.5
    y_scale = alt.Scale(domain=[_y_min, _y_max], zero=False)

    charts = []
    for i, param in enumerate(PARAMS):
        sub = agg[agg['parameter'] == param]
        ue_default_val = UNREAL_DEFAULTS.get(param)

        color_encoding = alt.Color(
            'resolution:N',
            scale=alt.Scale(domain=res_order, range=res_colors),
            sort=res_order,
            legend=alt.Legend(labelExpr="datum.value == '100%' ? '100%' : datum.value == '87%' ? '75%' : datum.value == '71%' ? '50%' : '25%'")
        )

        current_y_title = y_title if i == 0 or i == 2 else None

        base = alt.Chart(sub).encode(
            x=alt.X('value:Q', title=PARAM_LABELS.get(param, param)),
            color=color_encoding
        )

        band = base.mark_area(opacity=0.2, clip=True).encode(
            y=alt.Y('ci_low:Q', title=current_y_title, scale=y_scale),
            y2=alt.Y2('ci_high:Q')
        )

        line = base.mark_line(point=True, clip=True).encode(
            y=alt.Y('mean:Q', scale=y_scale, title=current_y_title)
        )

        layers = [band, line]

        param_recs = RECOMMENDATIONS.get(param, {})
        rec_data = []
        for res_lab, val in param_recs.items():
            rec_data.append({'resolution': reverse_map.get(res_lab, res_lab), 'val': val})

        side_label_data = []
        for res in sub['resolution'].unique():
            res_sub = sub[sub['resolution'] == res]
            opt_val = next((item['val'] for item in rec_data if item['resolution'] == res), None)

            if ue_default_val is not None:
                score_ue = res_sub.iloc[(res_sub['value'] - ue_default_val).abs().argsort()[:1]]['mean'].values[0]

                if annotation_mode == 'recs' and opt_val is not None:
                    score_opt = res_sub.iloc[(res_sub['value'] - opt_val).abs().argsort()[:1]]['mean'].values[0]
                    delta = score_opt - score_ue
                    side_label_data.append({
                        'Resolution': res,
                        'x': res_sub['value'].max(),
                        'y': res_sub.iloc[-1]['mean'],
                        'label': f"Δ={'+' if delta >= 0 else ''}{delta:.2f}",
                        'label2': None,
                    })

                elif annotation_mode == 'best_worst':
                    score_best  = res_sub['mean'].max()
                    score_worst = res_sub['mean'].min()
                    delta_best  = score_best  - score_ue
                    delta_worst = score_worst - score_ue
                    side_label_data.append({
                        'resolution': res,
                        'x': res_sub['value'].max(),
                        'y': res_sub.iloc[-1]['mean'],
                        'label':  f"↑{delta_best:+.2f}",
                        'label2': f"↓{delta_worst:+.2f}",
                    })

        if side_label_data:
            side_df = pd.DataFrame(side_label_data)
            res_rank = {'100%': 0, '87%': 1, '71%': 2, '50%': 3}
            side_df['rank'] = side_df['resolution'].map(res_rank)
            side_df = side_df.sort_values('rank')
            mid = (len(side_df) - 1) / 2

            dy_spacing = 45 if annotation_mode == 'best_worst' else 14

            for _, row in side_df.iterrows():
                dy_val = int((row['rank'] - mid) * dy_spacing)
            # for _, row in side_df.iterrows():
            #     dy_val = int((row['rank'] - mid) * 14)
                
                # First label (always shown)
                layers.append(
                    alt.Chart(pd.DataFrame([row])).mark_text(
                        align='left', dx=8, dy=dy_val - 12, fontSize=16, fontWeight='bold'
                    ).encode(x='x:Q', y='y:Q', text='label:N', color=color_encoding)
                )
                # Second label (only for best_worst mode)
                if annotation_mode == 'best_worst' and row['label2'] is not None:
                    layers.append(
                        alt.Chart(pd.DataFrame([row])).mark_text(
                            align='left', dx=8, dy=dy_val + 5, fontSize=16, fontWeight='bold'
                        ).encode(x='x:Q', y='y:Q', text='label2:N', color=color_encoding)
                    )

        if ue_default_val is not None:
            layers.append(alt.Chart(pd.DataFrame({'x': [ue_default_val]})).mark_rule(
                color='#FFD700', strokeDash=[4, 4], size=1.5, clip=True
            ).encode(x='x:Q'))

        charts.append(alt.layer(*layers).properties(width=200, height=200))

    top_row    = alt.hconcat(charts[0], charts[1])
    bottom_row = alt.hconcat(charts[2], charts[3])

    combined = alt.hconcat(top_row, bottom_row).resolve_scale(color='shared', y='shared').properties(title=plot_title)

    return combined.configure_axis(
        labelFontSize=16, titleFontSize=18, grid=True
    ).configure_legend(
        labelFontSize=16, titleFontSize=18, symbolOpacity=1
    ).configure_title(
        fontSize=16, anchor='middle'
    ).configure_view(stroke=None)

In [91]:
# 1. Generate the Raw CGVQM chart with Ideal lines
# pre-compute bounds so we can pass tight y limits
# agg_raw = (
#     df_raw.groupby(['parameter', 'resolution', 'value'])['score']
#     .agg(['mean', 'std'])
#     .reset_index()
# )
# raw_y_min = (agg_raw['mean'] - agg_raw['std']).min() - 0.5
# raw_y_max = (agg_raw['mean'] + agg_raw['std']).max() + 0.5

# raw_chart = plot_response_curves(
#     df_raw,
#     score_col='score',
#     y_title='Mean CGVQM Score',
#     plot_title='TAA Parameter Response Curves (Raw CGVQM)',
#     y_min=raw_y_min,
#     y_max=raw_y_max,
#     annotation_mode='recs'
# )

# 2. Generate the Mean-Centered chart with Ideal lines
agg_cent = (
    df_centered.groupby(['parameter', 'resolution', 'value'])['score_centered']
    .agg(['mean', 'std'])
    .reset_index()
)
cent_y_min = (agg_cent['mean'] - agg_cent['std']).min() - 0.5
cent_y_max = (agg_cent['mean'] + agg_cent['std']).max() + 0.5

cent_chart = plot_response_curves(
    df_centered,
    score_col='score_centered',
    y_title='Mean Centered CGVQM',
    plot_title='',
    y_min=cent_y_min,
    y_max=cent_y_max,
    annotation_mode='best_worst'
)

# 3. Combine and display
# alt.vconcat(raw_chart, cent_chart).resolve_scale(color='shared')
cent_chart.save('centered-scores.png', scale_factor=2.0)
# raw_chart.save('raw-scores.png', scale_factor=2.0)

#### Per Scene curves with Best and Worst Case

In [39]:
def plot_param_per_scene(input_df, param_name, score_col, y_title, plot_title, y_min=None, y_max=None):
    display_map = {'100%': '100%', '87%': '75%', '71%': '50%', '50%': '25%'}
    # param_name is now a variable (e.g., 'alpha_weight' or 'hist_weight')
    param = param_name 

    # 1. Data Cleaning
    scene_data = input_df[input_df['parameter'] == param].copy()
    scene_data['resolution'] = scene_data['resolution'].astype(str).str.replace('%%', '%')
    if not scene_data['resolution'].iloc[0].endswith('%'):
        scene_data['resolution'] = scene_data['resolution'] + '%'

    mean_agg = scene_data.groupby(['resolution', 'value'])[score_col].mean().reset_index(name='mean')

    _y_min = y_min if y_min is not None else scene_data[score_col].min() - 1
    _y_max = y_max if y_max is not None else scene_data[score_col].max() + 1
    y_scale = alt.Scale(domain=[_y_min, _y_max], zero=False)

    charts = []
    summary_stats = {} 

    for res_raw in ['100%', '87%', '71%', '50%']:
        res_label = display_map[res_raw]
        res_color = res_colors[res_order.index(res_raw)]
        # res_color = res_colors[res_order.index(res_label)]
        
        sub_scenes = scene_data[scene_data['resolution'] == res_raw]
        sub_mean = mean_agg[mean_agg['resolution'] == res_raw]
        
        if sub_scenes.empty: continue

        ue_default_val = UNREAL_DEFAULTS.get(param)
        opt_val = RECOMMENDATIONS.get(param, {}).get(res_label)

        x_axis_title = f"{PARAM_LABELS[param]}, Resolution {res_label}"

        # --- FIX: Added clip=True to marks ---
        scenes_layer = alt.Chart(sub_scenes).mark_line(
            opacity=0.15, strokeWidth=1, clip=True # Clips spaghetti lines
        ).encode(
            x=alt.X('value:Q', title=x_axis_title),
            y=alt.Y(f'{score_col}:Q', scale=y_scale, title=y_title if res_raw == '100%' else ""),
            detail='scene:N', color=alt.value(res_color)
        )
        
        mean_layer = alt.Chart(sub_mean).mark_line(
            strokeWidth=3, point=True, clip=True # Clips mean line
        ).encode(
            x='value:Q', y='mean:Q', color=alt.value(res_color)
        )

        # Rules (also clipped to frame)
        rules = []
        if ue_default_val is not None:
            rules.append(alt.Chart(pd.DataFrame({'x': [ue_default_val]})).mark_rule(
                color='#FFD700', strokeDash=[4, 4], size=1.5, clip=True
            ).encode(x='x:Q'))
        if opt_val is not None:
            rules.append(alt.Chart(pd.DataFrame({'x': [opt_val]})).mark_rule(
                color=res_color, strokeDash=[2, 2], size=1.5, clip=True
            ).encode(x='x:Q'))

        # Delta Calculation
        scene_list = sub_scenes['scene'].unique()
        deltas = []
        for s in scene_list:
            s_data = sub_scenes[sub_scenes['scene'] == s]
            if len(s_data) < 2: continue
            idx_opt = (s_data['value'] - opt_val).abs().idxmin()
            idx_ue  = (s_data['value'] - ue_default_val).abs().idxmin()
            deltas.append(s_data.loc[idx_opt, score_col] - s_data.loc[idx_ue, score_col])
        
        min_d, max_d = min(deltas), max(deltas)
        summary_stats[res_label] = {'min_delta': min_d, 'max_delta': max_d}

        # Multi-line Annotations (Not clipped so they stay visible on the right)
        anno_df = pd.DataFrame([{
            'x': sub_mean['value'].max(), 
            'y': sub_mean['mean'].iloc[-1], 
            'line1': f"↑{max_d:+.2f}",
            'line2': f"↓{min_d:+.2f}"
        }])
        
        text_max = alt.Chart(anno_df).mark_text(
            align='left', dx=10, dy=-7, fontSize=11, fontWeight='bold', color=res_color
        ).encode(x='x:Q', y='y:Q', text='line1:N')
        
        text_min = alt.Chart(anno_df).mark_text(
            align='left', dx=10, dy=7, fontSize=11, fontWeight='bold', color=res_color
        ).encode(x='x:Q', y='y:Q', text='line2:N')

        charts.append(alt.layer(scenes_layer, mean_layer, *rules, text_max, text_min).properties(
            width=230, height=200))

    final_chart = alt.hconcat(*charts).properties(title=plot_title).configure_axis(
        labelFontSize=12, titleFontSize=13, grid=True
    ).configure_title(fontSize=16, anchor='middle').configure_view(stroke=None)

    return final_chart, summary_stats

In [40]:
# Call for Alpha Weight
alpha_chart, alpha_stats = plot_param_per_scene(
    df_centered, 'alpha_weight', 'score_centered', 'Centered CGVQM', 'Alpha Weight'
)

# Call for History Weight
hist_chart, hist_stats = plot_param_per_scene(
    df_centered, 'hist_percent', 'score_centered', 'Centered CGVQM', 'History Percentage'
)

# alpha_chart.save('alpha-weight-allcurves.png', scale_factor=2.0)
# hist_chart.save('hist-percent-allcurves.png', scale_factor=2.0)

In [41]:
alpha_chart

alt.HConcatChart(...)

## Statistical Significance

In [29]:
from scipy import stats

results = []
for (param, res), group in df_centered.groupby(['parameter', 'resolution']):
    groups_by_value = [g['score_centered'].values 
                       for _, g in group.groupby('value')]
    
    anova_stat, anova_p = stats.f_oneway(*groups_by_value)
    kw_stat, kw_p       = stats.kruskal(*groups_by_value)
    
    results.append({
        'parameter':  param,
        'resolution': res,
        'anova_stat': round(anova_stat, 3),
        'anova_p':    round(anova_p,    4),
        'anova_sig':  anova_p < 0.05,
        'kw_stat':    round(kw_stat,    3),
        'kw_p':       round(kw_p,       4),
        'kw_sig':     kw_p < 0.05,
        'agree':      (anova_p < 0.05) == (kw_p < 0.05),
    })

results_df = pd.DataFrame(results).sort_values(['parameter', 'resolution'])
print(results_df.to_string(index=False))

   parameter  resolution  anova_stat  anova_p  anova_sig  kw_stat   kw_p  kw_sig  agree
alpha_weight          50      13.265   0.0000       True  152.861 0.0000    True   True
alpha_weight          71      11.658   0.0000       True  114.580 0.0000    True   True
alpha_weight          87      14.993   0.0000       True  131.402 0.0000    True   True
alpha_weight         100      15.036   0.0000       True  150.386 0.0000    True   True
 filter_size          50       0.303   0.9344      False    4.572 0.5998   False   True
 filter_size          71       0.103   0.9959      False    2.252 0.8951   False   True
 filter_size          87       0.127   0.9928      False    4.347 0.6298   False   True
 filter_size         100       4.290   0.0006       True   24.651 0.0004    True   True
hist_percent          50       2.924   0.0392       True   13.992 0.0029    True   True
hist_percent          71      22.916   0.0000       True   50.141 0.0000    True   True
hist_percent          87      31

In [48]:
plot_data = []
results_df = pd.DataFrame(results).sort_values(['parameter', 'resolution'])
for _, row in results_df.iterrows():
    def fmt_p(p):
        if p < 1e-10:
            return "≪0.001"
        elif p < 0.001:
            return f"{p:.2e}"
        else:
            return f"{p:.3f}"

    plot_data.append({
        'parameter':   row['parameter'],
        'resolution':  f"{row['resolution']}%",
        'log_p_kw':   -np.log10(max(row['kw_p'],    1e-300)),
        'log_p_anova':-np.log10(max(row['anova_p'], 1e-300)),
        'kw_p_str':    fmt_p(row['kw_p']),
        'anova_p_str': fmt_p(row['anova_p']),
    })

df_plot = pd.DataFrame(plot_data)

param_order  = ['alpha_weight', 'filter_size', 'hist_percent', 'num_samples']
param_labels = {
    'alpha_weight': 'Alpha weight', 'filter_size': 'Filter size',
    'hist_percent': 'History %',    'num_samples':  'Num samples',
}
df_plot['parameter_label'] = df_plot['parameter'].map(param_labels)
param_label_order = [param_labels[p] for p in param_order]
res_order_plot = ['50%', '71%', '87%', '100%']
res_display    = {'50%': '25%', '71%': '50%', '87%': '75%', '100%': '100%'}
df_plot['resolution_label'] = df_plot['resolution'].map(res_display)
res_label_order = [res_display[r] for r in res_order_plot]

def make_heatmap(df, color_col, text_col):
    param_label_map = {
        'Alpha weight': 'Alpha\nweight',
        'Filter size':  'Filter\nsize',
        'History %':    'History\n%',
        'Num samples':  'Num\nsamples',
    }
    df = df.copy()
    df['parameter_label'] = df['parameter_label'].map(param_label_map)
    stacked_label_order = [param_label_map[p] for p in param_label_order]

    heatmap = alt.Chart(df).mark_rect(stroke='white', strokeWidth=1.5).encode(
        x=alt.X('resolution_label:O',
                sort=res_label_order,
                title='Resolution',
                axis=alt.Axis(labelAngle=0, labelFontSize=12, titleFontSize=13)),
        y=alt.Y('parameter_label:O',
                sort=stacked_label_order,
                title=None,
                axis=alt.Axis(labelFontSize=12)),
        color=alt.Color(f'{color_col}:Q',
            scale=alt.Scale(scheme='orangered', domain=[0, 12], clamp=True),
            legend=alt.Legend(title='−log₁₀(p)', titleFontSize=11, labelFontSize=11, gradientLength=100)),
    )

    text = alt.Chart(df).mark_text(fontSize=11, fontWeight='bold').encode(
        x=alt.X('resolution_label:O', sort=res_label_order),
        y=alt.Y('parameter_label:O', sort=stacked_label_order),
        text=alt.Text(f'{text_col}:N'),
        color=alt.condition(
            alt.datum[color_col] > 6,
            alt.value('white'),
            alt.value('#333333')
        )
    )

    return (heatmap + text).properties(width=200, height=200)

kw_chart    = make_heatmap(df_plot, 'log_p_kw',    'kw_p_str')
# anova_chart = make_heatmap(df_plot, 'log_p_anova', 'anova_p_str', 'One-way ANOVA')

# chart = make_heatmap(df_plot, 'log_p_kw', 'kw_p_str', 
#                      'Kruskal–Wallis: parameter effect significance (centered CGVQM)')

kw_chart.configure_axis(
    grid=False, labelFontSize=12, titleFontSize=13
).configure_view(
    stroke=None
).show()

kw_chart.save("statsig.png", scale_factor = 2)

alt.LayerChart(...)

## Compute Time

## Content Dependence